In [1]:
import os
import sys

PROJECT_DIR = (
    "/work/archcomplab/ssaraf/"
    "Maya Archarological Sites/Model Training/"
    "swin3d_project"
)

os.chdir(PROJECT_DIR)

if PROJECT_DIR not in sys.path:
    sys.path.insert(
        0,
        PROJECT_DIR,
    )

print("Working directory:", os.getcwd())

Working directory: /work/archcomplab/ssaraf/Maya Archarological Sites/Model Training/swin3d_project


In [2]:
from pathlib import Path

import torch

from models.swin3d_model import Model
from datasets.belize_laz_dataset import create_dataloaders
from evaluation.evaluate_avd import evaluate_best_model_avd

/work/archcomplab/ssaraf/envs/maya_env/lib/python3.10/site-packages/MinkowskiEngine/__init__.py:36: UserWarning: The environment variable `OMP_NUM_THREADS` not set. MinkowskiEngine will automatically set `OMP_NUM_THREADS=16`. If you want to set `OMP_NUM_THREADS` manually, please export it on the command line before running a python script. e.g. `export OMP_NUM_THREADS=12; python your_program.py`. It is recommended to set it below 24.
  warnings.warn(


In [3]:
_, _, test_loader = create_dataloaders(
    split_dir="dataset_splits",
    batch_size=1,
    voxel_max=65_000,
    num_workers=4,
)

print("Test batches:", len(test_loader))

Test batches: 80


In [4]:
device = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

model = Model(
    num_classes=3,
    in_channels=4,
).to(device)

print("Device:", device)

Device: cuda


In [5]:
BEST_CHECKPOINT = Path(
    "/mnt/nrdstor/archcomplab/ssaraf/"
    "swin3d_checkpoints/"
    "dice_vicinity/"
    "swin3d_dice_vicinity_best_02.pth"
)

AVD_RESULTS_DIR = Path(
    "/mnt/nrdstor/archcomplab/ssaraf/"
    "swin3d_results/generalized_dice/dice_vicinity"
)

print("Checkpoint exists:", BEST_CHECKPOINT.exists())
print("Checkpoint:", BEST_CHECKPOINT)
print("Output:", AVD_RESULTS_DIR)

Checkpoint exists: True
Checkpoint: /mnt/nrdstor/archcomplab/ssaraf/swin3d_checkpoints/dice_vicinity/swin3d_dice_vicinity_best_02.pth
Output: /mnt/nrdstor/archcomplab/ssaraf/swin3d_results/generalized_dice/dice_vicinity


In [6]:
sample = test_loader.dataset[0]

print("coord:", sample["coord"].shape)
print("metric_coord:", sample["metric_coord"].shape)
print("feat:", sample["feat"].shape)
print("segment:", sample["segment"].shape)

assert sample["coord"].shape[0] == sample["metric_coord"].shape[0]
assert sample["coord"].shape[0] == sample["feat"].shape[0]
assert sample["coord"].shape[0] == sample["segment"].shape[0]

coord: torch.Size([65000, 3])
metric_coord: torch.Size([65000, 3])
feat: torch.Size([65000, 1])
segment: torch.Size([65000])


In [7]:
from torch.utils.data import DataLoader, Subset

small_test_dataset = Subset(
    test_loader.dataset,
    range(min(80, len(test_loader.dataset))),
)

small_test_loader = DataLoader(
    small_test_dataset,
    batch_size=1,
    shuffle=False,
    num_workers=0,
    pin_memory=True,
    collate_fn=test_loader.collate_fn,
)

print("Tiles in quick test:", len(small_test_dataset))

Tiles in quick test: 80


In [8]:
avd_output = evaluate_best_model_avd(
    model=model,
    test_loader=small_test_loader,
    checkpoint_path=BEST_CHECKPOINT,
    output_dir=AVD_RESULTS_DIR / "quick_test",
    device=device,

    cell_size_m=1.0,
    radii_m=(5.0,),
    min_points_per_cell=1,
    min_evidence_cells=3,
    beta=2.0,

    print_progress=True,
    save_csv=True,
)

/work/archcomplab/ssaraf/envs/maya_env/lib/python3.10/site-packages/torch/functional.py:534: UserWarning: torch.meshgrid: in an upcoming release, it will be required to pass the indexing argument. (Triggered internally at ../aten/src/ATen/native/TensorShape.cpp:3595.)
  return _VF.meshgrid(tensors, **kwargs)  # type: ignore[attr-defined]
/work/archcomplab/ssaraf/software/Swin3D/Swin3D/modules/swin3d_layers.py:239: UserWarning: The torch.cuda.*DtypeTensor constructors are no longer recommended. It's best to use methods such as torch.tensor(data, dtype=*, device='cuda') to create tensors. (Triggered internally at ../torch/csrc/tensor/python_tensor.cpp:78.)
  offset = torch.cuda.IntTensor(offset)


Evaluated AVD batch 1/80
Evaluated AVD batch 5/80
Evaluated AVD batch 10/80
Evaluated AVD batch 15/80
Evaluated AVD batch 20/80
Evaluated AVD batch 25/80
Evaluated AVD batch 30/80
Evaluated AVD batch 35/80
Evaluated AVD batch 40/80
Evaluated AVD batch 45/80
Evaluated AVD batch 50/80
Evaluated AVD batch 55/80
Evaluated AVD batch 60/80
Evaluated AVD batch 65/80
Evaluated AVD batch 70/80
Evaluated AVD batch 75/80
Evaluated AVD batch 80/80

AVD metric: cell=1 m, minimum evidence=3 cells, F_beta beta=2
class | radius_m | precision | recall | AVD-F_beta | near/total predicted cells | detected/total features
structure | 5 | 0.3341 | 0.2738 | 0.2841 | 1334/3993 | 23/84
plazuela | 5 | 0.5476 | 0.9670 | 0.8386 | 46230/84417 | 88/91
archaeological_discovery | 5 | 0.5662 | 0.9778 | 0.8537 | 49288/87057 | 88/90
macro AVD-F_beta @ 5 m = 0.6588

Checkpoint epoch: 55


In [9]:
avd_results_df = avd_output[
    "results_df"
]

display(avd_results_df)

,Mode / Class,Radius (m),Grid Cell Size (m),Minimum Points Per Cell,Minimum Evidence Cells,Beta,GT Features,Detected Features,Vicinity Recall,Vicinity Precision,AVD-F2,Predicted Cells,Useful Predicted Cells,False Predicted Cells
0,structure,5.0,1.0,1,3,2.0,84,23,0.273810,0.334085,0.284059,3993,1334,2659
1,plazuela,5.0,1.0,1,3,2.0,91,88,0.967033,0.547639,0.838590,84417,46230,38187
2,archaeological_discovery,5.0,1.0,1,3,2.0,90,88,0.977778,0.566158,0.853650,87057,49288,37769


In [10]:
avd_macro_df = avd_output[
    "macro_df"
]

display(avd_macro_df)

,Mode / Class,Radius (m),Structure AVD-F2,Plazuela AVD-F2,Macro AVD-F2
0,archaeological_macro,5.0,0.284059,0.83859,0.561325


In [11]:
display(
    avd_results_df[
        avd_results_df["Radius (m)"] == 5.0
    ]
)

,Mode / Class,Radius (m),Grid Cell Size (m),Minimum Points Per Cell,Minimum Evidence Cells,Beta,GT Features,Detected Features,Vicinity Recall,Vicinity Precision,AVD-F2,Predicted Cells,Useful Predicted Cells,False Predicted Cells
0,structure,5.0,1.0,1,3,2.0,84,23,0.273810,0.334085,0.284059,3993,1334,2659
1,plazuela,5.0,1.0,1,3,2.0,91,88,0.967033,0.547639,0.838590,84417,46230,38187
2,archaeological_discovery,5.0,1.0,1,3,2.0,90,88,0.977778,0.566158,0.853650,87057,49288,37769


In [10]:
_, _, test_loader = create_dataloaders(
    split_dir="dataset_splits",
    batch_size=1,
    voxel_max=65_000,
    num_workers=4,
)

In [11]:
avd_output = evaluate_best_model_avd(
    model=model,
    test_loader=test_loader,
    checkpoint_path=BEST_CHECKPOINT,
    output_dir=AVD_RESULTS_DIR / "radius_5m",
    device=device,

    cell_size_m=1.0,
    radii_m=(5.0,),
    min_points_per_cell=1,
    min_evidence_cells=3,
    beta=2.0,

    print_progress=True,
    save_csv=True,
)

KeyboardInterrupt: 